In [1]:
import numpy as np
import pandas as pd
import torch

from torch import nn
import torch.nn.functional as F

from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv

/Users/Mourya/miniforge3/envs/aircraft-conflict-gnn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_graphs = torch.load(
    "processed_data/train_graphs.pt",
    weights_only=False
)

val_graphs = torch.load(
    "processed_data/val_graphs.pt",
    weights_only=False
)

test_graphs = torch.load(
    "processed_data/test_graphs.pt",
    weights_only=False
)

print(len(train_graphs))
print(len(val_graphs))
print(len(test_graphs))

504
108
108


In [3]:
train_loader = DataLoader(
    train_graphs,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=8,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=8,
    shuffle=False
)

print("DataLoaders ready")

DataLoaders ready


In [4]:
class ConflictGCN(nn.Module):

    def __init__(self, node_dim=6, edge_dim=5, hidden_dim=64):
        super().__init__()

        self.conv1 = GCNConv(node_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)

        self.edge_mlp = nn.Sequential(
            nn.Linear(hidden_dim * 2 + edge_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, data):

        x = F.relu(self.conv1(data.x, data.edge_index))
        x = self.conv2(x, data.edge_index)

        src = x[data.edge_index[0]]
        dst = x[data.edge_index[1]]

        edge_input = torch.cat(
            [src, dst, data.edge_attr],
            dim=1
        )

        return self.edge_mlp(edge_input).squeeze(-1)

In [5]:
model = ConflictGCN()

print(model)

ConflictGCN(
  (conv1): GCNConv(6, 64)
  (conv2): GCNConv(64, 64)
  (edge_mlp): Sequential(
    (0): Linear(in_features=133, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=1, bias=True)
  )
)


In [6]:
device = torch.device("cpu")

model = model.to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

criterion = torch.nn.BCEWithLogitsLoss()

print("Device:", device)
print("Optimizer:", optimizer.__class__.__name__)
print("Loss:", criterion.__class__.__name__)

Device: cpu
Optimizer: Adam
Loss: BCEWithLogitsLoss


In [7]:
batch = next(iter(train_loader))

print("Nodes:", batch.x.shape)
print("Edges:", batch.edge_index.shape)
print("Edge features:", batch.edge_attr.shape)
print("Labels:", batch.y.shape)

Nodes: torch.Size([411, 6])
Edges: torch.Size([2, 10935])
Edge features: torch.Size([10935, 5])
Labels: torch.Size([10935])


In [8]:
batch = batch.to(device)

with torch.no_grad():
    logits = model(batch)

print("Logits shape:", logits.shape)
print("Labels shape:", batch.y.shape)

Logits shape: torch.Size([10935])
Labels shape: torch.Size([10935])


In [9]:
loss = criterion(
    logits,
    batch.y.float()
)

print("Initial loss:", loss.item())

Initial loss: 3.6059632301330566


In [10]:
optimizer.zero_grad()

logits = model(batch)

loss = criterion(
    logits,
    batch.y.float()
)

loss.backward()

optimizer.step()

print("Loss:", loss.item())
print("✅ First training step completed")

Loss: 3.6059632301330566
✅ First training step completed


In [11]:
def train_epoch(model, loader, optimizer, criterion, device):

    model.train()

    total_loss = 0.0

    for batch in loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        logits = model(batch)

        loss = criterion(logits, batch.y.float())

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [12]:
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

def evaluate(model, loader, device):

    model.eval()

    y_true = []
    y_pred = []
    y_prob = []

    with torch.no_grad():

        for batch in loader:

            batch = batch.to(device)

            logits = model(batch)

            probs = torch.sigmoid(logits)

            preds = (probs >= 0.5).long()

            y_true.extend(batch.y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())
            y_prob.extend(probs.cpu().numpy())

    return (
        np.array(y_true),
        np.array(y_pred),
        np.array(y_prob)
    )